In [2]:
from datetime import datetime
import openpyxl
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

## Euribor

In [ ]:
# Load macro control variables (currency pairs and commodities)
control_var_dir = Path('../scraping/output_data/control_variables')


In [4]:
# Load Euribor data (3-month) from all files in euribor folder
euribor_dir = control_var_dir / 'euribor'
euribor_files = sorted(euribor_dir.glob('*.csv'))

print(f"Found {len(euribor_files)} euribor files")

# Load and concatenate all euribor files
euribor_dfs = []
for file in euribor_files:
    try:
        df_eur = pd.read_csv(file)
        euribor_dfs.append(df_eur)
    except Exception as e:
        print(f"Error reading {file.name}: {e}")

# Concatenate all euribor data
df_euribor = pd.concat(euribor_dfs, ignore_index=True)

Found 28 euribor files


In [5]:
df_euribor.head()

,dundasChartControl1_DRG_DataRowGrouping1_label,dundasChartControl1_DRG_DataRowGrouping1_dundasChartControl1_DCG_Period1_label,dundasChartControl1_DRG_DataRowGrouping1_dundasChartControl1_DCG_Period1_Value_X,dundasChartControl1_DRG_DataRowGrouping1_dundasChartControl1_DCG_Period1_Value_Y
0,1 week,August,08/01/2012 00:00:00,0.099
1,1 week,August,08/02/2012 00:00:00,0.096
2,1 week,August,08/03/2012 00:00:00,0.097
3,1 week,August,08/06/2012 00:00:00,0.096
4,1 week,August,08/07/2012 00:00:00,0.096


In [6]:
# Clean up df_euribor
# Rename columns
df_euribor = df_euribor.rename(columns={
    'dundasChartControl1_DRG_DataRowGrouping1_dundasChartControl1_DCG_Period1_label': 'Date',
    'dundasChartControl1_DRG_DataRowGrouping1_dundasChartControl1_DCG_Period1_Value_X': 'euribor_type',
    'dundasChartControl1_DRG_DataRowGrouping1_dundasChartControl1_DCG_Period1_Value_Y': 'euribor_3m' 
})

# Filter for 3 month euribor only
df_euribor = df_euribor[df_euribor['euribor_type'] == '3 month']

# Convert Date to datetime (format: "4 Nov 2025")
df_euribor['Date'] = pd.to_datetime(df_euribor['Date'], format='%d %b %Y')

# Keep only Date and euribor_3m columns
df_euribor = df_euribor[['Date', 'euribor_3m']]

# Reset index
df_euribor = df_euribor.reset_index(drop=True)

# Save to pickle
output_path = Path('data/euribor_3m.pkl')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_euribor.to_pickle(output_path)

print(f"Saved cleaned euribor data to {output_path}")
df_euribor.head()

Saved cleaned euribor data to data/euribor_3m.pkl


,Date,euribor_3m
0,2013-01-31,0.232
1,2013-01-30,0.230
2,2013-01-29,0.226
3,2013-01-28,0.224
4,2013-01-25,0.214


In [7]:
df_euribor.shape

(3577, 2)

## Eurozone 10y yield

In [8]:
df_eur10y = pd.read_csv("../scraping/output_data/control_variables/eurozone_10y_yield_irt_euryld_d__custom_19607531_linear_2_0.csv")
df_eur10y.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,freq,Time frequency,yld_curv,Yield curve,maturity,Maturity,bonds,...,geo,Geopolitical entity (reporting),TIME_PERIOD,Time,OBS_VALUE,Observation value,OBS_FLAG,Observation status (Flag) V2 structure,CONF_STATUS,Confidentiality status (flag)
0,dataflow,ESTAT:IRT_EURYLD_D(1.0),Euro yield curves - daily data,D,Daily,INS_FWD,Instantaneous forward yield curve,Y10,Maturity: 10 years,CGB_EA,...,EA,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2012-01-02,NaN,5.56617,NaN,NaN,NaN,NaN,NaN
1,dataflow,ESTAT:IRT_EURYLD_D(1.0),Euro yield curves - daily data,D,Daily,INS_FWD,Instantaneous forward yield curve,Y10,Maturity: 10 years,CGB_EA,...,EA,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2012-01-03,NaN,5.61315,NaN,NaN,NaN,NaN,NaN
2,dataflow,ESTAT:IRT_EURYLD_D(1.0),Euro yield curves - daily data,D,Daily,INS_FWD,Instantaneous forward yield curve,Y10,Maturity: 10 years,CGB_EA,...,EA,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2012-01-04,NaN,5.64884,NaN,NaN,NaN,NaN,NaN
3,dataflow,ESTAT:IRT_EURYLD_D(1.0),Euro yield curves - daily data,D,Daily,INS_FWD,Instantaneous forward yield curve,Y10,Maturity: 10 years,CGB_EA,...,EA,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2012-01-05,NaN,5.84685,NaN,NaN,NaN,NaN,NaN
4,dataflow,ESTAT:IRT_EURYLD_D(1.0),Euro yield curves - daily data,D,Daily,INS_FWD,Instantaneous forward yield curve,Y10,Maturity: 10 years,CGB_EA,...,EA,"Euro area (EA11-1999, EA12-2001, EA13-2007, EA...",2012-01-06,NaN,5.85700,NaN,NaN,NaN,NaN,NaN


In [9]:
df_eur10y = df_eur10y[df_eur10y['maturity'] == 'Y10']
df_eur10y = df_eur10y[df_eur10y['Yield curve'] == 'Par yield curve']
df_eur10y = df_eur10y[["maturity", "TIME_PERIOD", "OBS_VALUE"]]

df_eur10y = df_eur10y.reset_index(drop=True)

df_eur10y.sort_values("TIME_PERIOD").head(10)

,maturity,TIME_PERIOD,OBS_VALUE
0,Y10,2012-01-02,4.16663
1,Y10,2012-01-03,4.20001
2,Y10,2012-01-04,4.24699
3,Y10,2012-01-05,4.33196
4,Y10,2012-01-06,4.36859
5,Y10,2012-01-09,4.36582
6,Y10,2012-01-10,4.35796
7,Y10,2012-01-11,4.26634
8,Y10,2012-01-12,4.11339
9,Y10,2012-01-13,4.18351


In [10]:
df_eur10y = df_eur10y[["TIME_PERIOD", "OBS_VALUE"]]
df_eur10y = df_eur10y.rename(columns={
    'TIME_PERIOD': 'Date',
    'OBS_VALUE': 'eurozone_10y_yield'
})

In [11]:
# Save to pickle
output_path = Path('data/eurozone_10y.pkl')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_eur10y.to_pickle(output_path)

print(f"Saved cleaned eurozone yield data to {output_path}")
df_eur10y.head()

Saved cleaned eurozone yield data to data/eurozone_10y.pkl


,Date,eurozone_10y_yield
0,2012-01-02,4.16663
1,2012-01-03,4.20001
2,2012-01-04,4.24699
3,2012-01-05,4.33196
4,2012-01-06,4.36859


## Finland 10y yield

In [12]:
df_finland_10y = pd.read_csv("../scraping/output_data/control_variables/finland_bond_yields_viitelainojen_korot_v2_en.csv")
df_finland_10y.head()

,txtb_row_title,title,txtb_currency,txtb_value
0,Period,31.12.2025,"Yield on goverment bonds, 5 year",2.61
1,Period,31.12.2025,"Yield on goverment bonds, 10 year",3.17
2,Period,31.12.2025,Loan period 30 Aug 2022 – 15 Apr 2027,2.06
3,Period,31.12.2025,Loan period 6 Sep 2017 – 15 Sep 2027,2.18
4,Period,31.12.2025,Loan period 7 Feb 2012 – 4 Jul 2028,2.20


In [13]:
df_finland_10y = df_finland_10y[df_finland_10y['txtb_currency'] == 'Yield on goverment bonds, 10 year']
df_finland_10y = df_finland_10y[["title", "txtb_currency", "txtb_value"]]

df_finland_10y = df_finland_10y.reset_index(drop=True)

In [14]:
df_finland_10y.sort_values("title").head(10)

,title,txtb_currency,txtb_value
2779,1.1.2015,"Yield on goverment bonds, 10 year",0.79
2021,1.1.2018,"Yield on goverment bonds, 10 year",NaN
1766,1.1.2019,"Yield on goverment bonds, 10 year",NaN
3344,1.10.2012,"Yield on goverment bonds, 10 year",1.77
3095,1.10.2013,"Yield on goverment bonds, 10 year",2.02
2845,1.10.2014,"Yield on goverment bonds, 10 year",1.04
2590,1.10.2015,"Yield on goverment bonds, 10 year",0.85
1828,1.10.2018,"Yield on goverment bonds, 10 year",0.75
1573,1.10.2019,"Yield on goverment bonds, 10 year",-0.25
1319,1.10.2020,"Yield on goverment bonds, 10 year",-0.32


In [15]:
df_finland_10y = df_finland_10y.dropna(subset=['txtb_value'])

In [16]:
df_finland_10y = df_finland_10y.rename(columns={
    'title': 'Date',
    'txtb_value': 'finland_10y_yield'
})

df_finland_10y = df_finland_10y[["Date", "finland_10y_yield"]]

# Save to pickle
output_path = Path('data/finland_10y.pkl')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_finland_10y.to_pickle(output_path)

print(f"Saved cleaned finland govt bond yield data to {output_path}")
df_finland_10y.head()

Saved cleaned finland govt bond yield data to data/finland_10y.pkl


,Date,finland_10y_yield
0,31.12.2025,3.17
1,30.12.2025,3.16
2,29.12.2025,3.15
3,23.12.2025,3.18
4,22.12.2025,3.22


## Finland Unemployment

In [17]:
df_finland_unemployment = pd.read_csv("../scraping/output_data/control_variables/finland_unemployment_001_135z_2025m11_20260115-130759.csv")
df_finland_unemployment.head()

,Month,"Unemployment rate, %","Unemployment rate, %, trend","Unemployment rate, %, seasonally adjusted"
0,2011M01,8.6,8.3,8.2
1,2011M02,8.5,8.3,8.1
2,2011M03,9.9,8.2,8.8
3,2011M04,8.4,8.1,7.7
4,2011M05,9.8,8.0,7.7


In [18]:
# Process unemployment data
# 1. Parse month column
df_finland_unemployment['year'] = df_finland_unemployment['Month'].str[:4].astype(int)
df_finland_unemployment['month'] = df_finland_unemployment['Month'].str[5:7].astype(int)

# 2. Calculate year-on-year change
df_finland_unemployment = df_finland_unemployment.sort_values(['year', 'month']).reset_index(drop=True)
df_finland_unemployment['unemp_rate_yoy_change'] = df_finland_unemployment['Unemployment rate, %'].diff(12)

# 3. Apply 1 month lag (data from month M is released on day 1 of month M+1)
# E.g., August data (month 8) gets reflected on September 1 (day 1 of month 9)
df_finland_unemployment['release_year'] = df_finland_unemployment['year']
df_finland_unemployment['release_month'] = df_finland_unemployment['month'] + 1
df_finland_unemployment['release_day'] = 1

# Handle month overflow (e.g., December -> January of next year)
df_finland_unemployment.loc[df_finland_unemployment['release_month'] > 12, 'release_year'] += 1
df_finland_unemployment.loc[df_finland_unemployment['release_month'] > 12, 'release_month'] -= 12

# Create release date
df_finland_unemployment['release_date'] = pd.to_datetime(
    df_finland_unemployment[['release_year', 'release_month', 'release_day']].rename(
        columns={'release_year': 'year', 'release_month': 'month', 'release_day': 'day'}
    )
)

df_finland_unemployment.head(15)

,Month,"Unemployment rate, %","Unemployment rate, %, trend","Unemployment rate, %, seasonally adjusted",year,month,unemp_rate_yoy_change,release_year,release_month,release_day,release_date
0,2011M01,8.6,8.3,8.2,2011,1,NaN,2011,2,1,2011-02-01
1,2011M02,8.5,8.3,8.1,2011,2,NaN,2011,3,1,2011-03-01
2,2011M03,9.9,8.2,8.8,2011,3,NaN,2011,4,1,2011-04-01
3,2011M04,8.4,8.1,7.7,2011,4,NaN,2011,5,1,2011-05-01
4,2011M05,9.8,8.0,7.7,2011,5,NaN,2011,6,1,2011-06-01
5,2011M06,8.7,8.0,8.3,2011,6,NaN,2011,7,1,2011-07-01
6,2011M07,7.1,7.9,8.1,2011,7,NaN,2011,8,1,2011-08-01
7,2011M08,6.6,7.8,7.5,2011,8,NaN,2011,9,1,2011-09-01
8,2011M09,7.0,7.7,7.9,2011,9,NaN,2011,10,1,2011-10-01
9,2011M10,7.1,7.7,7.8,2011,10,NaN,2011,11,1,2011-11-01


In [19]:
# Create daily time series
# Get the date range from the earliest to latest release date
min_date = df_finland_unemployment['release_date'].min()
max_date = df_finland_unemployment['release_date'].max()

# Create daily date range
daily_dates = pd.date_range(start=min_date, end=max_date, freq='D')
df_daily = pd.DataFrame({'Date': daily_dates})

# Prepare unemployment data for merging
df_unemp_for_merge = df_finland_unemployment[['release_date', 'unemp_rate_yoy_change']].copy()
df_unemp_for_merge = df_unemp_for_merge.rename(columns={
    'release_date': 'Date',
    'unemp_rate_yoy_change': 'unemp_rate_change_lagged'
})

# Drop NaN values (first 12 months won't have YoY change)
df_unemp_for_merge = df_unemp_for_merge.dropna()

# Merge and forward fill (each value stays constant until next release)
df_daily = df_daily.merge(df_unemp_for_merge, on='Date', how='left')
df_daily['unemp_rate_change_lagged'] = df_daily['unemp_rate_change_lagged'].ffill()

# Keep only Date and unemp_rate_change_lagged
df_unemployment_daily = df_daily[['Date', 'unemp_rate_change_lagged']].copy()

# Remove 2011 data
# df_unemployment_daily = df_unemployment_daily[df_unemployment_daily['Date'].dt.year > 2011]

# Save to pickle
output_path = Path('data/finland_unemployment_daily.pkl')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_unemployment_daily.to_pickle(output_path)

print(f"Saved daily unemployment data to {output_path}")
print(f"Date range: {df_unemployment_daily['Date'].min()} to {df_unemployment_daily['Date'].max()}")
print(f"Total days: {len(df_unemployment_daily)}")
df_unemployment_daily.tail(20)

Saved daily unemployment data to data/finland_unemployment_daily.pkl
Date range: 2011-02-01 00:00:00 to 2025-12-01 00:00:00
Total days: 5418


,Date,unemp_rate_change_lagged
5398,2025-11-12,1.5
5399,2025-11-13,1.5
5400,2025-11-14,1.5
5401,2025-11-15,1.5
5402,2025-11-16,1.5
5403,2025-11-17,1.5
5404,2025-11-18,1.5
5405,2025-11-19,1.5
5406,2025-11-20,1.5
5407,2025-11-21,1.5


In [20]:
df_unemployment_daily.isna().sum()

Date                          0
unemp_rate_change_lagged    365
dtype: int64

## Finland CPI

In [21]:
df_finland_cpi = pd.read_csv("../scraping/output_data/control_variables/finland_cpi_001_11xd_2025m12_20260115-131031.csv")
df_finland_cpi.head()

,Month,Commodity,Point figure,Annual change (%),Monthly change (%),Annual impact (%-units),Monthly impact (%-units)
0,2011M01,0 CONSUMER PRICE INDEX,101.78,3.03,0.36,3.037,0.364
1,2011M02,0 CONSUMER PRICE INDEX,102.40,3.32,0.60,3.329,0.609
2,2011M03,0 CONSUMER PRICE INDEX,102.96,3.30,0.54,3.300,0.546
3,2011M04,0 CONSUMER PRICE INDEX,103.16,3.16,0.19,3.159,0.194
4,2011M05,0 CONSUMER PRICE INDEX,103.21,3.34,0.04,3.344,0.048


In [22]:
# Process CPI data
# 1. Parse month column
df_finland_cpi['year'] = df_finland_cpi['Month'].str[:4].astype(int)
df_finland_cpi['month'] = df_finland_cpi['Month'].str[5:7].astype(int)

# 2. Use the Annual change (%) column which is already YoY change
df_finland_cpi = df_finland_cpi.sort_values(['year', 'month']).reset_index(drop=True)
df_finland_cpi['cpi_yoy_change'] = df_finland_cpi['Annual change (%)']

# 3. Apply 1 month lag (data from month M is released on day 15 of month M+1)
# E.g., August data (month 8) gets reflected on September 15 (day 15 of month 9)
df_finland_cpi['release_year'] = df_finland_cpi['year']
df_finland_cpi['release_month'] = df_finland_cpi['month'] + 1
df_finland_cpi['release_day'] = 15

# Handle month overflow (e.g., December -> January of next year)
df_finland_cpi.loc[df_finland_cpi['release_month'] > 12, 'release_year'] += 1
df_finland_cpi.loc[df_finland_cpi['release_month'] > 12, 'release_month'] -= 12

# Create release date
df_finland_cpi['release_date'] = pd.to_datetime(
    df_finland_cpi[['release_year', 'release_month', 'release_day']].rename(
        columns={'release_year': 'year', 'release_month': 'month', 'release_day': 'day'}
    )
)

df_finland_cpi.head(15)

,Month,Commodity,Point figure,Annual change (%),Monthly change (%),Annual impact (%-units),Monthly impact (%-units),year,month,cpi_yoy_change,release_year,release_month,release_day,release_date
0,2011M01,0 CONSUMER PRICE INDEX,101.78,3.03,0.36,3.037,0.364,2011,1,3.03,2011,2,15,2011-02-15
1,2011M02,0 CONSUMER PRICE INDEX,102.40,3.32,0.60,3.329,0.609,2011,2,3.32,2011,3,15,2011-03-15
2,2011M03,0 CONSUMER PRICE INDEX,102.96,3.30,0.54,3.300,0.546,2011,3,3.30,2011,4,15,2011-04-15
3,2011M04,0 CONSUMER PRICE INDEX,103.16,3.16,0.19,3.159,0.194,2011,4,3.16,2011,5,15,2011-05-15
4,2011M05,0 CONSUMER PRICE INDEX,103.21,3.34,0.04,3.344,0.048,2011,5,3.34,2011,6,15,2011-06-15
5,2011M06,0 CONSUMER PRICE INDEX,103.51,3.54,0.29,3.541,0.290,2011,6,3.54,2011,7,15,2011-07-15
6,2011M07,0 CONSUMER PRICE INDEX,103.22,3.95,-0.28,3.958,-0.280,2011,7,3.95,2011,8,15,2011-08-15
7,2011M08,0 CONSUMER PRICE INDEX,103.62,3.83,0.38,3.838,0.387,2011,8,3.83,2011,9,15,2011-09-15
8,2011M09,0 CONSUMER PRICE INDEX,104.05,3.69,0.41,3.697,0.414,2011,9,3.69,2011,10,15,2011-10-15
9,2011M10,0 CONSUMER PRICE INDEX,104.31,3.54,0.24,3.543,0.249,2011,10,3.54,2011,11,15,2011-11-15


In [23]:
# Create daily time series for CPI
# Get the date range from the earliest to latest release date
min_date = df_finland_cpi['release_date'].min()
max_date = df_finland_cpi['release_date'].max()

# Create daily date range
daily_dates = pd.date_range(start=min_date, end=max_date, freq='D')
df_daily_cpi = pd.DataFrame({'Date': daily_dates})

# Prepare CPI data for merging
df_cpi_for_merge = df_finland_cpi[['release_date', 'cpi_yoy_change']].copy()
df_cpi_for_merge = df_cpi_for_merge.rename(columns={
    'release_date': 'Date',
    'cpi_yoy_change': 'cpi_rate_change_lagged'
})

# Merge and forward fill (each value stays constant until next release)
df_daily_cpi = df_daily_cpi.merge(df_cpi_for_merge, on='Date', how='left')
df_daily_cpi['cpi_rate_change_lagged'] = df_daily_cpi['cpi_rate_change_lagged'].ffill()

# Keep only Date and cpi_rate_change_lagged
df_cpi_daily = df_daily_cpi[['Date', 'cpi_rate_change_lagged']].copy()

# Remove 2011 data
df_cpi_daily = df_cpi_daily[df_cpi_daily['Date'].dt.year > 2011]

# Save to pickle
output_path = Path('data/finland_cpi_daily.pkl')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_cpi_daily.to_pickle(output_path)

print(f"Saved daily CPI data to {output_path}")
print(f"Date range: {df_cpi_daily['Date'].min()} to {df_cpi_daily['Date'].max()}")
print(f"Total days: {len(df_cpi_daily)}")
df_cpi_daily.head(20)

Saved daily CPI data to data/finland_cpi_daily.pkl
Date range: 2012-01-01 00:00:00 to 2026-01-15 00:00:00
Total days: 5129


,Date,cpi_rate_change_lagged
320,2012-01-01,3.35
321,2012-01-02,3.35
322,2012-01-03,3.35
323,2012-01-04,3.35
324,2012-01-05,3.35
325,2012-01-06,3.35
326,2012-01-07,3.35
327,2012-01-08,3.35
328,2012-01-09,3.35
329,2012-01-10,3.35


## Finland Consumer Confidence

In [24]:
df_finland_consumer_conf = pd.read_csv("../scraping/output_data/control_variables/finland_consumer_confidence_001_13ai_2025m12_20260115-131154.csv")
df_finland_consumer_conf.head()

,Month,Code,Balance figure,Outlook,Monthly change,Annual change
0,2011M01,"A1 Consumer confidence indicator, CCI = (B1+B2...",0.3,2,1.8,-1.4
1,2011M02,"A1 Consumer confidence indicator, CCI = (B1+B2...",0.4,2,0.1,-3.2
2,2011M03,"A1 Consumer confidence indicator, CCI = (B1+B2...",-0.2,3,-0.6,-3.4
3,2011M04,"A1 Consumer confidence indicator, CCI = (B1+B2...",-0.6,3,-0.4,-3.9
4,2011M05,"A1 Consumer confidence indicator, CCI = (B1+B2...",-1.5,4,-0.9,-1.8


In [25]:
# Process Consumer Confidence data
# 1. Parse month column
df_finland_consumer_conf['year'] = df_finland_consumer_conf['Month'].str[:4].astype(int)
df_finland_consumer_conf['month'] = df_finland_consumer_conf['Month'].str[5:7].astype(int)

# 2. Use the Annual change column for YoY change
df_finland_consumer_conf = df_finland_consumer_conf.sort_values(['year', 'month']).reset_index(drop=True)
df_finland_consumer_conf['consumer_conf_yoy_change'] = df_finland_consumer_conf['Annual change']

# 3. Apply 1 month lag (data from month M is released on day 1 of month M+1)
# E.g., August data (month 8) gets reflected on September 1 (day 1 of month 9)
df_finland_consumer_conf['release_year'] = df_finland_consumer_conf['year']
df_finland_consumer_conf['release_month'] = df_finland_consumer_conf['month'] + 1
df_finland_consumer_conf['release_day'] = 1

# Handle month overflow (e.g., December -> January of next year)
df_finland_consumer_conf.loc[df_finland_consumer_conf['release_month'] > 12, 'release_year'] += 1
df_finland_consumer_conf.loc[df_finland_consumer_conf['release_month'] > 12, 'release_month'] -= 12

# Create release date
df_finland_consumer_conf['release_date'] = pd.to_datetime(
    df_finland_consumer_conf[['release_year', 'release_month', 'release_day']].rename(
        columns={'release_year': 'year', 'release_month': 'month', 'release_day': 'day'}
    )
)

df_finland_consumer_conf.head(15)

,Month,Code,Balance figure,Outlook,Monthly change,Annual change,year,month,consumer_conf_yoy_change,release_year,release_month,release_day,release_date
0,2011M01,"A1 Consumer confidence indicator, CCI = (B1+B2...",0.3,2,1.8,-1.4,2011,1,-1.4,2011,2,1,2011-02-01
1,2011M02,"A1 Consumer confidence indicator, CCI = (B1+B2...",0.4,2,0.1,-3.2,2011,2,-3.2,2011,3,1,2011-03-01
2,2011M03,"A1 Consumer confidence indicator, CCI = (B1+B2...",-0.2,3,-0.6,-3.4,2011,3,-3.4,2011,4,1,2011-04-01
3,2011M04,"A1 Consumer confidence indicator, CCI = (B1+B2...",-0.6,3,-0.4,-3.9,2011,4,-3.9,2011,5,1,2011-05-01
4,2011M05,"A1 Consumer confidence indicator, CCI = (B1+B2...",-1.5,4,-0.9,-1.8,2011,5,-1.8,2011,6,1,2011-06-01
5,2011M06,"A1 Consumer confidence indicator, CCI = (B1+B2...",-5.8,5,-4.3,-7.4,2011,6,-7.4,2011,7,1,2011-07-01
6,2011M07,"A1 Consumer confidence indicator, CCI = (B1+B2...",-5.5,5,0.3,-8.1,2011,7,-8.1,2011,8,1,2011-08-01
7,2011M08,"A1 Consumer confidence indicator, CCI = (B1+B2...",-8.9,5,-3.4,-12.6,2011,8,-12.6,2011,9,1,2011-09-01
8,2011M09,"A1 Consumer confidence indicator, CCI = (B1+B2...",-9.5,5,-0.6,-14.5,2011,9,-14.5,2011,10,1,2011-10-01
9,2011M10,"A1 Consumer confidence indicator, CCI = (B1+B2...",-9.3,5,0.2,-12.1,2011,10,-12.1,2011,11,1,2011-11-01


In [26]:
# Create daily time series for Consumer Confidence
# Get the date range from the earliest to latest release date
min_date = df_finland_consumer_conf['release_date'].min()
max_date = df_finland_consumer_conf['release_date'].max()

# Create daily date range
daily_dates = pd.date_range(start=min_date, end=max_date, freq='D')
df_daily_conf = pd.DataFrame({'Date': daily_dates})

# Prepare consumer confidence data for merging
df_conf_for_merge = df_finland_consumer_conf[['release_date', 'consumer_conf_yoy_change']].copy()
df_conf_for_merge = df_conf_for_merge.rename(columns={
    'release_date': 'Date',
    'consumer_conf_yoy_change': 'consumer_conf_change_lagged'
})

# Merge and forward fill (each value stays constant until next release)
df_daily_conf = df_daily_conf.merge(df_conf_for_merge, on='Date', how='left')
df_daily_conf['consumer_conf_change_lagged'] = df_daily_conf['consumer_conf_change_lagged'].ffill()

# Keep only Date and consumer_conf_change_lagged
df_consumer_conf_daily = df_daily_conf[['Date', 'consumer_conf_change_lagged']].copy()

# Remove 2011 data
df_consumer_conf_daily = df_consumer_conf_daily[df_consumer_conf_daily['Date'].dt.year > 2011]

# Save to pickle
output_path = Path('data/finland_consumer_conf_daily.pkl')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_consumer_conf_daily.to_pickle(output_path)

print(f"Saved daily consumer confidence data to {output_path}")
print(f"Date range: {df_consumer_conf_daily['Date'].min()} to {df_consumer_conf_daily['Date'].max()}")
print(f"Total days: {len(df_consumer_conf_daily)}")
df_consumer_conf_daily.head(20)

Saved daily consumer confidence data to data/finland_consumer_conf_daily.pkl
Date range: 2012-01-01 00:00:00 to 2026-01-01 00:00:00
Total days: 5115


,Date,consumer_conf_change_lagged
334,2012-01-01,-8.9
335,2012-01-02,-8.9
336,2012-01-03,-8.9
337,2012-01-04,-8.9
338,2012-01-05,-8.9
339,2012-01-06,-8.9
340,2012-01-07,-8.9
341,2012-01-08,-8.9
342,2012-01-09,-8.9
343,2012-01-10,-8.9
